In [1]:
%load_ext autoreload
%autoreload 2

# BiCodec Audio Tokenizer Example

This notebook demonstrates the BiCodec audio tokenizer

**Key Properties**
- Input sample rate: 16 kHz
- Output sample rate: 16 kHz (24 kHz option but 50 TPS fixed)
- Codebook size: 8192 tokens
- Token rate: 50 Hz (50 tokens per second fixed)
- Architecture: Wav2Vec Encoder for Semantic tokens & TDNN for Global Tokens + Decoder
- Supports speech, audio, and music

## Setup and Imports


In [2]:
import sys
import os
import torch
import numpy as np
from datasets import load_dataset
from IPython.display import Audio, display
import matplotlib.pyplot as plt

# Add our tokenizer module to path
sys.path.append('../src')

# Import our tokenizer registry
from audio_tokenizers import get_tokenizer

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


/users/arsaikia/benchmark-audio-tokenizer/.venv-bicodec/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch version: 2.6.0a0+df5bbc09d1.nv24.11
CUDA available: True
Using device: cuda


## Load ESB Diagnostic AMI Dataset


In [3]:
# Load the AMI clean split
print("Loading ESB diagnostic dataset - AMI clean split...")
dataset = load_dataset('esb/diagnostic-dataset', 'ami', split='clean')

print(f"Dataset loaded with {len(dataset)} samples")
print(f"Sample keys: {dataset[0].keys()}")


Loading ESB diagnostic dataset - AMI clean split...


Dataset loaded with 770 samples


/usr/local/lib/python3.12/dist-packages/librosa/core/intervals.py:8: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Sample keys: dict_keys(['audio', 'ortho_transcript', 'norm_transcript', 'id', 'dataset'])


## Find Long Audio Samples


In [4]:
# Calculate duration for each sample and find the longest ones
durations = []
for i, sample in enumerate(dataset):
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    duration = len(audio_array) / sr
    durations.append((i, duration, sample['norm_transcript']))

# Sort by duration and get top 5 longest
durations.sort(key=lambda x: x[1], reverse=True)
long_samples = durations[:5]

print("Top 5 longest audio samples:")
for idx, (i, dur, transcript) in enumerate(long_samples):
    print(f"{idx+1}. Sample {i}: {dur:.2f} seconds")
    print(f"   Transcript: {transcript[:100]}..." if len(transcript) > 100 else f"   Transcript: {transcript}")
    print()


Top 5 longest audio samples:
1. Sample 85: 11.77 seconds
   Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c ...

2. Sample 1: 11.70 seconds
   Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would lik...

3. Sample 100: 11.69 seconds
   Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go ...

4. Sample 31: 11.44 seconds
   Transcript: and also we talked about um a location function where maybe you could press a button on the t v and ...

5. Sample 210: 11.33 seconds
   Transcript: the the problem is if you have to go across the building and it adds some overhead every time you wa...



## Initialize BiCodec

In [5]:
# Initialize the tokenizer
print("Loading BiCodec tokenizer...")
tokenizer = get_tokenizer('bicodec', device=device, use_speaker_encoder=False)

print(f"Tokenizer loaded: {tokenizer}")
print(f"  Input sample rate: {tokenizer.sample_rate} Hz")
print(f"  Output sample rate: {tokenizer.output_sample_rate} Hz")
print(f"  Codebook size: {tokenizer.codebook_size:,}")
print(f"  Downsample rate: {tokenizer.downsample_rate}x")
print(f"  Token rate: ~{tokenizer.sample_rate / tokenizer.downsample_rate:.1f} Hz")


Loading BiCodec tokenizer...


Fetching 31 files: 100%|██████████| 31/31 [00:00<00:00, 1149.43it/s]
/usr/local/lib/python3.12/dist-packages/torch/nn/utils/weight_norm.py:143: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


Missing tensor: mel_transformer.spectrogram.window
Missing tensor: mel_transformer.mel_scale.fb
Tokenizer loaded: BiCodecAudioTokenizer(checkpoint='None', device='cuda', sample_rate=16000)
  Input sample rate: 16000 Hz
  Output sample rate: 16000 Hz
  Codebook size: 8,192
  Downsample rate: 320.0x
  Token rate: ~50.0 Hz


## Tokenize and Reconstruct Audio Samples


In [7]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "128"

# Process the top 3 longest samples
results = []

for idx in range(min(3, len(long_samples))):
    sample_idx, duration, transcript = long_samples[idx]
    sample = dataset[sample_idx]
    
    print(f"\n{'='*60}")
    print(f"Processing Sample {idx+1} (index {sample_idx})")
    print(f"Duration: {duration:.2f} seconds")
    print(f"Transcript: {transcript[:150]}..." if len(transcript) > 150 else f"Transcript: {transcript}")
    
    # Get audio data
    audio_array = sample['audio']['array']
    sr = sample['audio']['sampling_rate']
    
    # Convert to tensor and ensure correct shape (1, T) for WavTokenizer
    audio_tensor = torch.from_numpy(audio_array).float()
    if audio_tensor.dim() == 1:
        audio_tensor = audio_tensor.unsqueeze(0)  # (T,) -> (1, T)
    
    # Tokenize
    print("\nTokenizing...")
    tokens, encode_info = tokenizer.encode(audio_tensor, sr=sr)

    # Show token information
    print(f"Semantic token shape: {encode_info["semantic_token_shape"]}")
    print(f"Global token shape: {encode_info["global_token_shape"]}")
    print(f"Number of tokens: {encode_info["num_tokens"]}")
    print(f"Tokens per second: {encode_info["num_tokens"] / duration:.1f}")
    
    # Show first 20 discrete token values
    semantic_tokens = tokens
    global_tokens = getattr(semantic_tokens, "global_tokens")
    token_values = semantic_tokens.squeeze().cpu().numpy()
    print(f"\nFirst 20 discrete token IDs:")
    print(token_values[:20])
    print(f"Token ID range: [{token_values.min()}, {token_values.max()}]")
    print(f"Unique tokens used: {len(np.unique(token_values))}")
    
    # Decode back to audio
    print("\nDecoding tokens back to audio...")
    reconstructed, decode_info = tokenizer.decode(tokens)
    
    print(f"Reconstructed shape: {reconstructed.shape}")
    print(f"Output sample rate: {decode_info['output_sample_rate']} Hz")
    
    # Handle shape: (1, 1, T) or (1, T) or (T,)
    rec = reconstructed
    if rec.dim() == 3:
        rec = rec[0, 0] if rec.shape[1] == 1 else rec[0]
    elif rec.dim() == 2:
        rec = rec[0]
    rec_np = rec.detach().cpu().numpy() if torch.is_tensor(rec) else rec
    
    # Store results
    results.append({
        'original': audio_array,
        'original_sr': sr,
        'reconstructed': rec_np,
        'reconstructed_sr': decode_info['output_sample_rate'],
        'tokens': token_values,
        'transcript': transcript,
        'duration': duration
    })
    
print(f"\n{'='*60}")
print("Processing complete!")



Processing Sample 1 (index 85)
Duration: 11.77 seconds
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you...

Tokenizing...
Semantic token shape: [1, 588]
Global token shape: [1, 1, 32]
Number of tokens: 620
Tokens per second: 52.7

First 20 discrete token IDs:
[5668 1621  949 6801 3851  106  281 5667 6767 5585 5467 2776 7650 2117
 6775 2097 4915 5188 7330 4765]
Token ID range: [2, 8177]
Unique tokens used: 570

Decoding tokens back to audio...
Reconstructed shape: torch.Size([1, 1, 188160])
Output sample rate: 16000 Hz

Processing Sample 2 (index 1)
Duration: 11.70 seconds
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokenizing...
Semantic token shape: [1, 584]
Global token shape: [1, 1, 32]
Number of tokens: 616
Tokens per second: 52.6

First 20 discrete token IDs:
[ 294 

## Play Original vs Reconstructed Audio


In [8]:
# Display audio players for comparison
for idx, result in enumerate(results):
    print(f"\n{'='*60}")
    print(f"Sample {idx+1} - Duration: {result['duration']:.2f}s")
    print(f"Transcript: {result['transcript'][:200]}..." if len(result['transcript']) > 200 else f"Transcript: {result['transcript']}")
    print(f"\nTokens used: {len(result['tokens'])} ({len(result['tokens'])/result['duration']:.1f} tokens/sec)")
    
    print(f"\nOriginal Audio ({result['original_sr']} Hz):")
    display(Audio(result['original'], rate=result['original_sr']))
    
    print(f"\nReconstructed Audio ({result['reconstructed_sr']} Hz):")
    display(Audio(result['reconstructed'], rate=result['reconstructed_sr']))
    
    # Compression ratio
    original_size = len(result['original']) * 2  # 16-bit audio
    token_size = len(result['tokens']) * 2  # 12-bit tokens (log2(4096) = 12)
    compression_ratio = original_size / token_size
    print(f"\nCompression ratio: {compression_ratio:.1f}x")



Sample 1 - Duration: 11.77s
Transcript: so that people can go arou go back and forth and choose if or or then again if you just i guess i c i can see like some kind of a thing where like you sort of have like the number come up on the t v l...

Tokens used: 588 (50.0 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 320.3x

Sample 2 - Duration: 11.70s
Transcript: so i guess we have to reflect on our experiences with remote controls to decide what um we would like to see in a convenient practical

Tokens used: 584 (49.9 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 320.5x

Sample 3 - Duration: 11.69s
Transcript: that f the fruit and vegetable theme is the is the current trend and and therefore um we need to go for that if we want

Tokens used: 584 (50.0 tokens/sec)

Original Audio (16000 Hz):



Reconstructed Audio (16000 Hz):



Compression ratio: 320.3x


## Summary Statistics


In [ ]:
# Calculate summary statistics
print("Summary of Tokenization Results:")
print("="*60)

total_duration = sum(r['duration'] for r in results)
total_tokens = sum(len(r['tokens']) for r in results)

print(f"Total audio processed: {total_duration:.2f} seconds")
print(f"Total tokens generated: {total_tokens:,}")
print(f"Average tokens per second: {total_tokens/total_duration:.1f}")

# Calculate average compression ratio
avg_compression = np.mean([
    (len(r['original']) * 2) / (len(r['tokens']) * 2) 
    for r in results
])
print(f"Average compression ratio: {avg_compression:.1f}x")

print(f"\nBiCodec Properties:")
print(f"  Bitrate: {50 * 13 / 1000:.2f} kbps (approx)")
print(f"  Token rate: 50 Hz")
print(f"  Bits per token: 13 (log2(8192))")
print(f"  Codebook size: 8192")
print(f"  Input: 16 kHz mono")
print(f"  Output: 16 kHz mono")

Summary of Tokenization Results:
Total audio processed: 35.16 seconds
Total tokens generated: 1,756
Average tokens per second: 49.9
Average compression ratio: 320.4x

BiCodec Properties:
  Bitrate: 0.65 kbps (approx)
  Token rate: 50 Hz
  Bits per token: 13 (log2(8192))
  Codebook size: 8192
  Input: 16 kHz mono
  Output: 16 kHz mono
